In [0]:
dbutils.widgets.text("env", "dev", "env")
dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "all_pending_process", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "scheduled", "run type")

In [0]:
global env, param_name, process_name, process_date,workspace, metadata_schema
env = dbutils.widgets.get("env")
param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
workspace = 'workspace'
metadata_schema = f"metadata_{env}"
print(env)
print(param_name)
print(process_name)
print(process_date)
print(workspace)
print(metadata_schema)
print(run_type)

In [0]:

metadata_tbl = ["audit_log","dp_batch","dp_calender","dp_metadata","param_table","table_column_metadata"]

def run_metadata_notebook(env, workspace, timeout=300):
    result = dbutils.notebook.run(
        "../metadata/metadata_table",
        timeout,
        {
            "env": f"{env}",
            "workspace": f"{workspace}"
        }
    )
    return result

In [0]:
# from pyspark.sql.utils import AnalysisException

missing_items = []

# Check if schema exists
schemas = [row['databaseName'] for row in spark.sql(f"SHOW DATABASES in {workspace}").collect()]
print(schemas)
if metadata_schema not in schemas:
    missing_items.append(f"Schema '{metadata_schema}'")

# Check if tables exist
else:
    tables = [row['tableName'] for row in spark.sql(f"SHOW TABLES IN {metadata_schema}").collect()]
    print(tables)
    for tbl in metadata_tbl:
        tbl = tbl+f"_{env}"
        if tbl not in tables:
            missing_items.append(f"Table '{tbl}' in schema '{metadata_schema}'")

if missing_items:
    for item in missing_items:
        print(f"Missing: {item}")
    run_metadata_notebook(env, workspace)

In [0]:
def insert_log(param_name, process_name, source_schema, source_table, target_schema, target_table, process_date,process_status, log):
    # error_msg = str(log).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{process_name}', '{source_schema}', '{source_table}', '{target_schema}', '{target_table}', '{process_date}', current_timestamp(), '{process_status}', '{log}')
        """
    )

In [0]:
def get_param_details():
    try:
        df_param_table = spark.sql(f"select * from {metadata_schema}.param_table_{env} where param_name='{param_name}'")
        records = df_param_table.toPandas().to_dict('records')
        if not records:
            print("No records found for the given param_name.")
            insert_log(param_name, process_name, 'param_table_check','param_table_check','param_table_check','param_table_check',process_date,'success','No records found for the given param_name.')
            return None
        insert_log(param_name, process_name, 'param_table_check','param_table_check','param_table_check','param_table_check',process_date,'success','no logs generated')
        return records[0]
    except Exception as e:
        print(f"Error fetching param details: {e}")
        error_msg = str(e).replace("'", "''")
        insert_log(param_name, process_name, 'param_table_check','param_table_check','param_table_check','param_table_check',process_date,'failed',error_msg)
        return None

#fetching process details from metadata table
# param_details = get_param_details()
# print(param_details)

In [0]:
def get_process_details():
    try:
        df_process_details = spark.sql(
            f"select * from {metadata_schema}.dp_metadata_{env} where param_name='{param_name}'"
        )
        records = df_process_details.toPandas().to_dict('records')
        if not records:
            print("No records found for the given param_name and process_name.")
            insert_log(param_name, process_name, 'dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check',process_date,'success','No records found for the given param_name and process_name.')
            return None
        insert_log(param_name, process_name, 'dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check',process_date,'success','no logs generated')
        return records
    except Exception as e:
        print(f"Error fetching process details: {e}")
        error_msg = str(e).replace("'", "''")
        insert_log(param_name, process_name, 'dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check','dp_metadata_table_check',process_date,'failed',error_msg)
        return None

# process_details = get_process_details()
# print(process_details)

In [0]:
def get_daily_run_status():
    try:
        df_daily_run_status = spark.sql(
            f"select * from {metadata_schema}.dp_batch_{env} where param_name='{param_name}' and process_date = '{process_date}'"
        )
        records = df_daily_run_status.toPandas().to_dict('records')
        if not records:
            print("No records found for the given param_name and process_name.")
            insert_log(param_name, process_name, 'dp_batch_table_check','dp_batch_table_check','dp_batch_table_check','dp_batch_table_check',process_date,'success','No records found for the given param_name and process_name.')
            return None
        insert_log(param_name, process_name, 'dp_batch_table_check','dp_batch_table_check','dp_batch_table_check','dp_batch_table_check',process_date,'success','no logs generated')
        return records
    except Exception as e:
        print(f"Error fetching daily run status: {e}")
        error_msg = str(e).replace("'", "''")
        insert_log(param_name, process_name, 'dp_batch_table_check','dp_batch_table_check','dp_batch_table_check','dp_batch_table_check',process_date,'failed',error_msg)
        return None

# daily_run_status = get_daily_run_status()
# print(daily_run_status)

In [0]:
param_details = get_param_details()
process_details = get_process_details()
daily_run_status = get_daily_run_status()

In [0]:
print(daily_run_status)
print(process_details)
print(param_details)

In [0]:
def get_pending_details(daily_run_status,process_details):
    succeded_process_list = []
    for i in daily_run_status:
        if i['process_date'] == process_date and i['process_status'] == 'success' and i['param_name'] == param_name:
            succeded_process_list.append(i['process_name'])
        else:
            continue
    pending_process_list = []
    for i in process_details:
        if i['param_name'] == param_name:
            if i['process_name'] not in succeded_process_list:
                pending_process_list.append(i)
        else:
            continue
    insert_log(param_name, process_name, 'pending_process_check','pending_process_check','pending_process_check','pending_process_check',process_date,'success','no logs generated')
    return pending_process_list
    

In [0]:
pending_process_list = get_pending_details(daily_run_status,process_details)

In [0]:
if run_type == 'scheduled':
    if len(pending_process_list) > 0:
        print("Pending process found")
        # print(pending_process_list)
        for i in pending_process_list:
            comment = f"""running {i['process_name']} process for process_date : {str(process_date)}"""
            insert_log(param_name, process_name, 'run_process','run_process','run_process','run_process',process_date,'success',comment)
            print(comment)
            # NOTEBOOK RUN ADDED IN FUTURE FULL / LOAD
        comment = f"""running {i['process_name']} process for process_date : {str(process_date)}"""
        insert_log(param_name, process_name, 'run_param_date_update','run_param_date_update','run_param_date_update','run_param_date_update',process_date,'success','no logs available')
        print("Param date update notebook Run starts")
        # PARAM_DATE UPDATE RUN ADDED IN FUTURE
        insert_log(param_name, process_name, 'param_date_update_completed','param_date_update_completed','param_date_update_completed','param_date_update_completed',process_date,'success','param_date_update_completed')
if run_type == 'force_run':
    comment = "force run will perform for process: "+process_name
    print(comment)
    insert_log(param_name, process_name, 'run_process','run_process','run_process','run_process',process_date,'success',comment)
    # NOTEBOOK RUN ADDED IN FUTURE ROLLBACK
    # NOTEBOOK RUN ADDED IN FUTURE FULL / LOAD
# PARAM_DATE UPDATE RUN WILL NOT HAPPEN IN FORCE RUN
    

In [0]:
dbutils.notebook.exit("success")

In [0]:
%sql
select * from metadata_dev.audit_log_dev